# TER Picopatt - Analyse exploratoire

Importation des librairies principales et définition des dossiers de travail.

Les fonctions communes utilisées pour lire et préparer les données sont regroupées dans `src/picopatt/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import picopatt as fc

# Dossiers de données et de sortie 
DATA_DIR = Path("dataproc")  
DATA_EXPORT_DIR = Path("data/processed/analyse_exploratoire")
FIG_DIR = Path("analyse_exploratoire/figures")

# Création automatique des dossiers
fc.create_folder(DATA_EXPORT_DIR)
fc.create_folder(FIG_DIR)

# Vérification
print(" Environnement prêt :")
print(f" DATA_DIR   -> {DATA_DIR}")
print(f" DATA_EXPORT_DIR -> {DATA_EXPORT_DIR}")
print(f" FIG_DIR    -> {FIG_DIR}")


pd.set_option("display.max_columns", 200)

Lecture de tous les fichiers de données (.csv, .xlsx) du dossier `dataproc` dans un seul tableau, nettoyage des colonnes, puis vérification de la couverture temporelle et la répartition des mesures et ajout des colonnes `M_slot` (créneau horaire) et `date`.

In [ ]:
raw = fc.load_all(DATA_DIR)

# Résumé
print("\nChargement terminé.")
print("Couverture :", raw['date'].min(), "->", raw['date'].max())
print("Parcours :", raw['track_id'].dropna().unique())
print("\nNombre de mesures par M_slot et parcours :")
print(
    raw.pivot_table(index="track_id", columns="M_slot", values="fichier_originaire", aggfunc="count")
       .fillna(0)
       .astype(int)
)

Les données couvrent du `29 octobre au 16 janvier` soit environ 3 mois.

On a trois parcours de collecte : `Antigone, Boulevards et Écusson`.

On observe également `quatre passages par jour` (matin, midi, après-midi et soir), presque `tout les mardis et jeudis`.

Globalement, la base de données est bien équilibrée entre les parcours et les passages.

Cependant, on remarque que le parcours Écusson contient un peu moins de données, ce qui peut indiquer que certains passages n’ont pas été effectués, qu’un problème de mesure est survenu ou tout simplement moins de mesures ont été éffectues pour Écusson.

In [ ]:
# Répertoire d’enregistrement des figures
COMPT = FIG_DIR / "comptage"

# Sélection uniquement des variables météorologiques
# On filtre les colonnes contenant des données climatiques (tair, rh, sw, lw, tmrt, pet, etc.)
meteo_cols = [col for col in raw.columns if any(key in col for key in ["tair", "rh", "sw", "lw", "tmrt", "pet"])]

# Calcul du taux de valeurs manquantes
missing = raw[meteo_cols].isna().mean().sort_values(ascending=False) * 100

print("Taux de valeurs manquantes (%) pour les variables météorologiques :")
display(missing.head(20))

# Comptage global par parcours
print("Nombre total de points de mesure par parcours :")
display(raw["track_id"].value_counts())

# Visualisation graphique
plt.figure(figsize=(10, 4))
plt.bar(missing.index, missing.values)
plt.title("Taux de valeurs manquantes (%) - Variables météorologiques")
plt.xticks(rotation=90)
plt.ylabel("% de valeurs manquantes")
plt.tight_layout()
plt.savefig(COMPT / "na_pct_vars_meteo.png", dpi=150)
plt.show()

In [ ]:
DATA_NOZERO = Path("data/processed/picopatt/clean_nozeros")
bd = fc.load_all(DATA_NOZERO, False)

# Sélection uniquement des variables météorologiques
meteo_cols = [col for col in bd.columns if any(key in col for key in ["tair", "rh", "sw", "lw", "tmrt", "pet"])]

if not meteo_cols:
    raise ValueError("Aucune variable météorologique détectée dans les fichiers chargés.")

# Calcul du taux de valeurs manquantes
missing = bd[meteo_cols].isna().mean().sort_values(ascending=False) * 100

print("\nTaux de valeurs manquantes (%) pour les variables météorologiques :")
display(missing)

# Visualisation graphique
plt.figure(figsize=(10, 4))
plt.bar(missing.index, missing.values)
plt.title("Taux de valeurs manquantes (%) - Variables météorologiques (clean_nozeros)")
plt.xticks(rotation=90)
plt.ylabel("% de valeurs manquantes")
plt.tight_layout()
plt.savefig(COMPT / "na_pct_vars_meteoclean.png", dpi=150)
plt.show()

Les variables climatiques présentent très peu de valeurs manquantes, ce qui garantit une base de données fiable et exploitable.

Certaines colonnes pour les sections sont quasiment vides, ce qui est normal compte tenu de la configuration du jeu de données, et elles pourront donc être ignorées lors des analyses.

On observe également le nombre total de relevés par parcours, avec Écusson qui présente le volume le plus faible.

Le bloc suivant affiche une `heatmap` montrant le nombre d’échantillons collectés pour chaque combinaison `parcours × M_slot` (M1 à M4).  
Cela permet d’identifier les périodes ou parcours avec plus ou moins de données.

In [ ]:
# Tableau de comptage des échantillons
tab = (
    raw.dropna(subset=["track_id", "M_slot"])
       .pivot_table(
           index="track_id",
           columns="M_slot",
           values="timestamp",
           aggfunc="count",
           observed=True
       )
       .reindex(columns=["M1", "M2", "M3", "M4"])
       .fillna(0)
       .astype(int)
)

# Heatmap
plt.figure(figsize=(6, 4))
plt.imshow(tab.values, cmap="viridis", aspect="auto") 
plt.xticks(range(tab.shape[1]), tab.columns)
plt.yticks(range(tab.shape[0]), tab.index)
plt.title("Nombre d'échantillons - parcours x mesure")
plt.colorbar(label="Nombre d’échantillons")
plt.tight_layout()

# Sauvegarde et affichage
plt.show()

Le parcours `Antigone` est le plus stable, possède le plus grand nombre de mesures sur l’ensemble des créneaux.

Le parcours `Boulevards` présente davantage de données sur le créneau M2 (midi), mais moins sur M1 et M3 (après-midi).
	
Le parcours `Écusson` contient globalement de bonnes mesures mais beaucoup moins sur M4.


# Pourquoi `Écusson` a moins de données ?

In [ ]:
# Période de mesure et nombre de jours distincts
periode_parcours = (
    raw.dropna(subset=["track_id", "timestamp"])
       .groupby("track_id")
       .agg(
           date_debut=("timestamp", "min"),
           date_fin=("timestamp", "max"),
           nb_jours=("date", "nunique")
       )
       .reset_index()
)

# Nombre total de passages (jour + M_slot) par parcours
nb_passages_par_parcours = (
    raw.dropna(subset=["track_id", "M_slot", "date"])
       .groupby("track_id")[["date", "M_slot"]]
       .apply(lambda x: x.drop_duplicates(subset=["date", "M_slot"]).shape[0])
       .reset_index(name="nb_passages")
)

# Détail des passages par créneau horaire (M1, M2, M3, M4)
nb_passages_par_Mslot = (
    raw.dropna(subset=["track_id", "M_slot", "date"])
       .groupby(["track_id", "M_slot"])
       .apply(lambda x: x.drop_duplicates(subset=["date", "M_slot"]).shape[0])
       .reset_index(name="nb_passages")
       .pivot(index="track_id", columns="M_slot", values="nb_passages")
       .fillna(0)
       .astype(int)
)

# Fusion des résultats en un tableau unique
synthese_passages = (
    periode_parcours
    .merge(nb_passages_par_parcours, on="track_id", how="left")
    .merge(nb_passages_par_Mslot, on="track_id", how="left")
)

display(synthese_passages)

# Nombre de passages par parcours et créneau horaire
plt.figure(figsize=(8, 5))
nb_passages_par_Mslot.plot(
    kind="bar",
    figsize=(8, 5),
    rot=0,
    title="Nombre de passages distincts par parcours et créneau horaire"
)
plt.xlabel("Parcours")
plt.ylabel("Nombre de passages (jour + créneau horaire)")
plt.tight_layout()
plt.savefig(COMPT/"nb_passages_distincts_par_Mslot.png", dpi=150)
plt.show()

In [ ]:
# Uniquement les fichiers liés à Écusson
paths_ecusson = sorted([p for p in DATA_DIR.rglob("*ecusson*") if p.suffix.lower() in (".csv", ".xlsx", ".xls")])

def get_dates_from_file(p):
    try:
        if p.suffix.lower() in (".xlsx", ".xls"):
            df = pd.read_excel(p)
        else:
            df = pd.read_csv(p, sep=None, engine="python") 
        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", dayfirst=True)
            return sorted(df["timestamp"].dropna().dt.date.unique())
        else:
            return []
    except Exception as e:
        print(f" Erreur lecture {p.name}: {e}")
        return []

file_dates = pd.DataFrame([
    {"fichier": p.name, "dates_trouvées": get_dates_from_file(p)}
    for p in paths_ecusson
])

display(file_dates)

# Combien de jours distincts au total :
all_dates = sorted({d for sublist in file_dates["dates_trouvées"] for d in sublist})
print(f"\n Nombre total de jours uniques mesurés pour le parcours Écusson : {len(all_dates)}")

Le parcours `Antigone` a bénéficié de la couverture la plus complète, tandis que `Écusson` présente le moins de passages. Cela s'explique par le fait qu'`Écusson` est bénéficié d'une journée de collecte de moins que `Antigone` et `Boulevards` 

In [ ]:
df_ecusson = raw[raw["track_id"] == "ecusson"]
sections_lues = sorted(df_ecusson["section_id"].dropna().unique())

# Le parcours doit avoir 75 sections
sections_theoriques = list(range(1, 76))

# On cherche les manquantes
sections_manquantes = sorted(set(sections_theoriques) - set(sections_lues))

print("Nombre total attendu :", len(sections_theoriques))
print("Nombre réellement lus :", len(sections_lues))
print("Sections manquantes :", sections_manquantes)

Nous savons que le nombre de sections pour le parcours `Écusson` est de 73.

Les sections 12 et 32 sont manquantes dans tout les fichiers d'`Écusson`. 

# Statistiques descriptives sur les variables météo

Calcul des statistiques globales, par parcours et par créneau horaire sur les variables météorologiques présentes dans les données nettoyées (dossier `clean_nozeros` issue du notebook `NoZero.ipynb`).

In [ ]:
# Liste des variables météo présentes (sans sw_back et sw_down)
CANDIDATES = [
    "tair_thermohygro","tair_tc1","tair_tc2","tair_anemo",
    "rh_thermohygro","ws","wdir",
    "sw_up","sw_front","sw_right",
    "lw_up","lw_down","lw_front","lw_back","lw_left","lw_right",
    "tmrt","pet"
]
METEO = [c for c in CANDIDATES if c in bd.columns]
num_cols = [c for c in METEO if c != "wdir"] 

# Résumé
print("Chargement terminé.")
print("Couverture :", bd['date'].min(), "->", bd['date'].max())
print("Parcours :", bd['track_id'].dropna().unique())
print("\nNombre de mesures par M_slot et parcours :")
print(
    bd.pivot_table(index="track_id", columns="M_slot", values="fichier_originaire", aggfunc="count")
       .fillna(0)
       .astype(int)
)

In [ ]:
stats_global = fc.summary_stats(bd, num_cols)
if "wdir" in METEO:
    stats_global.loc["wdir", "mean"] = fc.circular_mean_deg(bd["wdir"])

STATS = DATA_EXPORT_DIR / "stats"
fc.create_folder(STATS)
stats_global.to_csv(STATS / "stats_global.xlsx")
display(stats_global)

# Statistiques descriptives des variables météorologiques par parcours

In [ ]:
# Statistiques par parcours
track_code = {
    "antigone": "a",
    "boulevard": "b",
    "ecusson": "e"
}

by_track = []
for t, g in bd.groupby("track_id", dropna=True):
    s = fc.summary_stats(g, num_cols)
    if "wdir" in METEO:
        s.loc["wdir", "mean"] = fc.circular_mean_deg(g["wdir"])
    s = s.reset_index(names="variable")

    # Ajout de la lettre du parcours à chaque variable
    suffix = track_code.get(t, t[:1].lower())
    s["variable"] = s["variable"] + f"_{suffix}"

    by_track.append(s)

stats_by_track = pd.concat(by_track, ignore_index=True)
stats_by_track.to_csv(STATS / "stats_par_parcours.xlsx", index=False)
display(stats_by_track)

# Statistiques descriptives des variables météorologiques par parcours et par créneau horaire

In [ ]:
# Statistiques par parcours et créneau horaire
by_mslot = []
for (t, mslot), g in bd.groupby(["track_id", "M_slot"], dropna=True):
    s = fc.summary_stats(g, num_cols)
    if "wdir" in METEO and "wdir" in g.columns:
        s.loc["wdir", "mean"] = fc.circular_mean_deg(g["wdir"])
    s = s.reset_index(names="variable")

    # Ajout du suffixe parcours + Mslot 
    # ex: 'v'_aM1
    suffix = track_code.get(t, t[:1].lower())
    s["variable"] = s["variable"] + f"_{suffix}{mslot}"

    by_mslot.append(s)

stats_track_M = pd.concat(by_mslot, ignore_index=True)

# Tri des variables par parcours et mslot
stats_track_M["sort_key"] = stats_track_M["variable"].str.extract(r"_(.M\d)").astype(str)
stats_track_M = stats_track_M.sort_values("variable").drop(columns="sort_key")

stats_track_M.to_csv(STATS / "stats_par_parcours_par_mesure.xlsx", index=False)
display(stats_track_M)

# Matrices de corrélation par parcours

Affiche les `corrélations entre les variables météorologiques` pour chaque parcours.  

In [ ]:
CORR = FIG_DIR / "correlation"
fc.create_folder(CORR)

for t, g in bd.groupby("track_id"):
    sel = g[num_cols].select_dtypes(include=[np.number]).dropna(how="all", axis=1)
    
    if sel.shape[1] < 2: 
        print(f"Parcours {t} : pas assez de colonnes numériques pour corrélation.")
        continue

    corr = sel.corr()

    plt.figure(figsize=(9, 6))
    plt.title(f"Corrélations - {t}")
    plt.imshow(corr, aspect="auto")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.colorbar(label="r")
    plt.tight_layout()
    plt.savefig(CORR / f"corr_{t}.png", dpi=150)
    plt.show()

Sur Boulevards et Antigone, les températures mesurées par les différents capteurs sont fortement corrélées, traduisant une évolution cohérente du signal thermique. À l’inverse, l’Écusson présente des corrélations plus faibles, révélant un micro-environnement urbain plus hétérogène. Les composantes du rayonnement solaire sont fortement corrélées entre elles et avec les indices de confort thermique (tmrt, PET), soulignant le rôle central du rayonnement solaire, tandis que le rayonnement infrarouge montre des relations plus homogènes et stables entre parcours.


# Boxplots

Ce bloc affiche des `boxplots` pour comparer la distribution de chaques variables météorologiques clés entre les différents parcours .

In [ ]:
BOXPLOT = FIG_DIR / "boxplot"
fc.create_folder(BOXPLOT)

flierprops = dict(marker='o', markerfacecolor='black', markersize=2, linestyle='none', alpha=0.5)

for v in METEO:
    plt.figure(figsize=(7, 4))
    raw.boxplot(column=v, by="track_id", grid=False, flierprops=flierprops)
    plt.title(f"{v} par parcours") 
    plt.suptitle("")  
    plt.xlabel("Parcours")
    plt.ylabel(v)
    plt.tight_layout()
    plt.savefig(BOXPLOT / f"box{v}_parcours.png", dpi=150)
    plt.show()

## Distribution des variables météo par section au sein de chaque parcours

Ce bloc génère des **Boxplots** pour chaque variable météorologique, visualisant sa distribution pour chaque `section_id` au sein de chaque `track_id` (parcours). Cela permet d'identifier les variations des conditions météo le long de chaque parcours et de détecter d'éventuels anomalies ou spécificités liées à des sections particulières.

In [ ]:
BOXPLOT_BY_SECTION = FIG_DIR / "boxplot_by_section"
fc.create_folder(BOXPLOT_BY_SECTION)

ANT = BOXPLOT_BY_SECTION / "antigone"
BOU = BOXPLOT_BY_SECTION / "boulevards"
ECU = BOXPLOT_BY_SECTION / "ecusson"

fc.create_folder(ANT)
fc.create_folder(BOU)
fc.create_folder(ECU)

flierprops = dict(marker='o', markerfacecolor='red', markersize=3, linestyle='none', alpha=0.5)

for track_id, track_data in raw.groupby("track_id"):
    track_dir = BOXPLOT_BY_SECTION / track_id

    print(f"Genere des boxplots pour les parcours: {track_id}")

    for v in METEO:
        plt.figure(figsize=(10, 6))
        track_data.boxplot(column=v, by="section_id", grid=False, flierprops=flierprops, rot=90)
        plt.title(f"Distribution de {v} par section pour {track_id}")
        plt.suptitle("") 
        plt.xlabel("Section ID")
        plt.ylabel(v)
        plt.tight_layout()
        plt.savefig(track_dir / f"box_{v}_by_section_{track_id}.png", dpi=150)
        plt.show()

##  Distribution des variables météo par parcours

Ce bloc affiche des **violinPlots**  pour comparer la distribution de chauques variables météorologiques clés entre les différents parcours (`track_id`).

In [ ]:
VIOLIN_DIR = FIG_DIR / "violinplot"
fc.create_folder(VIOLIN_DIR)

for v in METEO:
    plt.figure(figsize=(7, 4))
    sns.violinplot(x="track_id", y=v, data=raw)
    plt.title(f"Distribution de {v} par parcours")
    plt.xlabel("Parcours")
    plt.ylabel(v)
    plt.tight_layout()
    plt.savefig(VIOLIN_DIR / f"violin_{v}_parcours.png", dpi=150)
    plt.show()

# Comparaison des Boxplots et des violinPlots

In [ ]:
COMPARE_PLOTS_DIR = FIG_DIR / "compare_box_violin"
fc.create_folder(COMPARE_PLOTS_DIR)

flierprops = dict(marker='o', markerfacecolor='red', markersize=3, linestyle='none', alpha=0.5)

for v in METEO:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    fig.suptitle(f"Distribution de {v} par parcours: Boxplot vs Violinplot")

    # Boxplot
    raw.boxplot(column=v, by="track_id", grid=False, flierprops=flierprops, ax=axes[0])
    axes[0].set_title("Boxplot")
    axes[0].set_xlabel("Parcours")
    axes[0].set_ylabel(v)
    fig.canvas.manager.set_window_title('') 

    # Violinplot
    sns.violinplot(x="track_id", y=v, data=raw, ax=axes[1], inner='quartile') 
    axes[1].set_title("Violinplot")
    axes[1].set_xlabel("Parcours")
    axes[1].set_ylabel(v) 

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
    plt.savefig(COMPARE_PLOTS_DIR / f"compare_{v}_parcours.png", dpi=150)
    plt.show()

# Distribution des variables météo

Ces blocs tracent un `histogramme` pour chaque variable météo sur différentes échelles (plus ou moins local), afin de visualiser la `répartition des valeurs mesurées` (et détecter d’éventuels biais ou valeurs extrêmes).

In [ ]:
# Distribution globale de chaque variable
DISTRIB = FIG_DIR / "distribution"
fc.create_folder(DISTRIB)

DISTRIB_G = DISTRIB / "distribution_globale"
fc.create_folder(DISTRIB_G)
for v in METEO:
    s = bd[v].dropna()
    if len(s) == 0:
        continue

    plt.figure(figsize=(6, 4))
    plt.hist(s, bins=50)
    plt.title(f"Distribution - {v}")
    plt.xlabel(v)
    plt.ylabel("Occurrence")
    plt.tight_layout()
    plt.savefig(DISTRIB_G / f"dist_global_{v}.png", dpi=150)

In [ ]:
# Distribution de chaque variable par parcours
DISTRIB_P = DISTRIB / "distribution_parcours"
fc.create_folder(DISTRIB_P)

for track in sorted(bd["track_id"].dropna().unique()):
    sub = bd[bd["track_id"] == track]
    
    if sub.empty:
        continue

    OUT_TRACK = DISTRIB_P / track
    fc.create_folder(OUT_TRACK)

    for v in METEO:
        s = sub[v].dropna()
        if len(s) == 0:
            continue

        plt.figure(figsize=(6, 4))
        plt.hist(s, bins=50)
        plt.title(f"Distribution - {v} ({track})")
        plt.xlabel(v)
        plt.ylabel("Occurrence")
        plt.tight_layout()
        plt.savefig(OUT_TRACK / f"dist_{track}_{v}.png", dpi=150)

In [ ]:
import math
import gc
import matplotlib
matplotlib.use("Agg")
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colorbar import ColorbarBase

DISTRIB_VAR = DISTRIB / "distributions_variables"
fc.create_folder(DISTRIB_VAR)

COL_MEAN_TRACK = "#E63946"
COL_MEAN_SEC   = "#1D3557"
COL_MEDIAN     = "#F4A261"

cmap = LinearSegmentedColormap.from_list("temp_scale", ["#1D4E89", "white", "#E63946"])

fig, axes = None, None

for track in sorted(bd["track_id"].dropna().unique()):
    sub_track = bd[bd["track_id"] == track]
    sections = sorted(sub_track["section_id"].dropna().unique())
    n_sec = len(sections)

    section_data = {
        sec: sub_track.loc[sub_track["section_id"] == sec]
        for sec in sections
    }

    print(f"Génération des distributions pour {track} ({n_sec} sections)")

    for v in METEO:
        data_all = sub_track[v].dropna()
        if len(data_all) == 0:
            continue

        VAR_DIR = DISTRIB_VAR / v
        fc.create_folder(VAR_DIR)

        # Pré-calcul bornes et statistiques globales
        xmin, xmax = data_all.min(), data_all.max()
        mean_val = data_all.mean()
        median_val = data_all.median()
        std_val = data_all.std()

        # Pré-calcul du y_max
        y_max = 0
        hist_data = {}
        for sec, df in section_data.items():
            s = df[v].dropna()
            if len(s) == 0:
                continue
            counts, bins = np.histogram(s, bins=50, range=(xmin, xmax))
            hist_data[sec] = (s, counts, bins)
            y_max = max(y_max, counts.max())
        y_max *= 1.1

        # Création figure une seule fois
        ncols = min(8, n_sec)
        nrows = math.ceil(n_sec / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.2))
        axes = np.array(axes).reshape(-1)

        # Titre global
        fig.text(0.5, 0.97, f"{track.capitalize()} — {v}",
                 ha="center", va="top", fontsize=16, fontweight="bold")

        for i, sec in enumerate(sections):
            ax = axes[i]
            if sec not in hist_data:
                ax.axis("off")
                continue

            s, counts, bins = hist_data[sec]
            ax.hist(s, bins=50, range=(xmin, xmax), color="skyblue", edgecolor="grey")

            sec_mean = s.mean()
            delta = (sec_mean - mean_val) / std_val

            intensity = min(abs(delta) * 0.5, 0.35)
            if delta > 0:
                ax.set_facecolor((1, 1 - intensity, 1 - intensity))
            else:
                ax.set_facecolor((1 - intensity, 1 - intensity * 0.6, 1))

            ax.axvline(mean_val, color=COL_MEAN_TRACK, linestyle="--", linewidth=1.3)
            ax.axvline(sec_mean, color=COL_MEAN_SEC, linestyle="-", linewidth=1.3)
            ax.axvline(median_val, color=COL_MEDIAN, linestyle=":", linewidth=1.3)

            ax.text(0.98, 0.9, f"{mean_val:.1f}", transform=ax.transAxes,
                    ha="right", va="top", fontsize=6, color=COL_MEAN_TRACK)
            ax.text(0.98, 0.8, f"{sec_mean:.1f}", transform=ax.transAxes,
                    ha="right", va="top", fontsize=6, color=COL_MEAN_SEC)
            ax.text(0.98, 0.7, f"{median_val:.1f}", transform=ax.transAxes,
                    ha="right", va="top", fontsize=6, color=COL_MEDIAN)

            ax.set_title(f"S{int(sec)}", fontsize=8)
            ax.set_xlim(xmin, xmax)
            ax.set_ylim(0, y_max)
            ax.tick_params(axis="both", labelsize=6)

        for j in range(i + 1, len(axes)):
            axes[j].axis("off")

        # Légende
        line_legend = [
            Line2D([0], [0], color=COL_MEAN_TRACK, linestyle="--", lw=1.5, label="Moyenne parcours"),
            Line2D([0], [0], color=COL_MEAN_SEC, linestyle="-", lw=1.5, label="Moyenne section"),
            Line2D([0], [0], color=COL_MEDIAN, linestyle=":", lw=1.5, label="Médiane parcours")
        ]
        fig.legend(handles=line_legend, loc='upper center', bbox_to_anchor=(0.5, 0.942),
                   ncol=3, fontsize=9, frameon=False)

        # Barre de dégradé
        cax = fig.add_axes([0.35, 0.905, 0.3, 0.012])
        norm = plt.Normalize(-2, 2)
        cb = ColorbarBase(cax, cmap=cmap, norm=norm, orientation='horizontal')
        cb.set_ticks([-2, -1, 0, 1, 2])
        cb.ax.tick_params(labelsize=7)
        cb.set_label("Écart à la moyenne du parcours", fontsize=8, labelpad=-1)

        fig.subplots_adjust(top=0.87, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.3)
        outfile = VAR_DIR / f"dist_{v}_{track}.png"
        fig.savefig(outfile, dpi=120, bbox_inches="tight")
        plt.close(fig)
        gc.collect()


# Profil des variables selon le moment de la journée (M1–M4)

Ce bloc trace, pour chaque variable météo et pour chaques parcours (`Antigone`, `Boulevards`, `Écusson`), l’évolution moyenne des valeurs selon les quatre créneaux horaires de mesure (`M1` à `M4`).

In [ ]:
PROFIL = FIG_DIR / "profil"
fc.create_folder(PROFIL)

PROFIL_MEAN_MSLOTS = PROFIL / "comparer_moyenne_mslot"
fc.create_folder(PROFIL_MEAN_MSLOTS)

order = ["M1", "M2", "M3", "M4"]
agg = bd.groupby(["track_id", "M_slot"])[METEO].mean().reset_index()
tracks = ["antigone", "boulevards", "ecusson"]

for v in METEO:
    plt.figure(figsize=(7, 4))
    for t in tracks:
        gg = agg[agg.track_id == t].set_index("M_slot").reindex(order)[v]
        if gg.isna().all():
            continue
        plt.plot(order, gg.values, marker="o", label=t)
    
    plt.title(f"Profil {v}")
    plt.xlabel("Créneau horaire")
    plt.ylabel(v)
    plt.legend()
    plt.tight_layout()
    plt.savefig(PROFIL_MEAN_MSLOTS / f"profil_{v}_Mslots.png", dpi=150)

Maintenant, on souhaite voir un profil pour chaque variable de chaque mesure 

In [ ]:
import re
import matplotlib
matplotlib.use("Agg")  
import gc

PROFIL_POINT = FIG_DIR / "profil" / "point"
fc.create_folder(PROFIL_POINT)

fig, ax = plt.subplots(figsize=(9, 4))

for file_name, gfile in bd.groupby("fichier_originaire"):
    m = re.search(r"_(\d{8})_(\d{4})", file_name)

    date_str, time_str = m.groups()
    date_obj = pd.to_datetime(date_str, format="%Y%m%d")
    jour = date_obj.day
    mois = date_obj.strftime("%b").lower()

    track = gfile["track_id"].dropna().iloc[0]
    mslot = gfile["M_slot"].dropna().iloc[0] if "M_slot" in gfile.columns else "UNK"

    OUT_TRACK = PROFIL_POINT / track
    fc.create_folder(OUT_TRACK)

    FOLDER = OUT_TRACK / f"{jour}{mois}_{mslot}"
    fc.create_folder(FOLDER)

    print(f"Génération des profils pour {track} ({mslot}, {jour} {mois})")

    gfile = gfile.sort_values("point_id")

    xticks = np.linspace(0, len(gfile), 6).astype(int)

    for v in METEO:
        if v not in gfile.columns:
            continue
        y = gfile[v].values
        x = gfile["point_id"].values
        if np.all(np.isnan(y)):
            continue

        ax.cla() 
        ax.plot(x, y, color="royalblue", lw=1.2, alpha=0.9)

        ax.set_title(f"{track.capitalize()} — {v} — {mslot} ({jour}/{mois})",
                     fontsize=14, fontweight="bold")
        ax.set_xlabel("point_id")
        ax.set_ylabel(v)
        ax.grid(alpha=0.3)
        ax.set_xticks(xticks)

        outfile = FOLDER / f"profil_{v}_{track}_{jour}{mois}_{mslot}.png"
        fig.tight_layout()
        fig.savefig(outfile, dpi=120, bbox_inches="tight")

    plt.close('all')
    gc.collect()


On compare les différentes dates de mesures selon leur Mslot, afin d'observer les tendances de parcours

In [ ]:
import re
import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
import gc

COMPARE_M = FIG_DIR / "profil" / "comparer_meme_Mslot"
fc.create_folder(COMPARE_M)

cmap = get_cmap("tab10")

for track in sorted(bd["track_id"].dropna().unique()):
    sub_track = bd[bd["track_id"] == track]
    OUT_TRACK = COMPARE_M / track
    fc.create_folder(OUT_TRACK)

    for mslot in sorted(sub_track["M_slot"].dropna().unique()):
        sub_slot = sub_track[sub_track["M_slot"] == mslot]
        OUT_M = OUT_TRACK / mslot
        fc.create_folder(OUT_M)

        print(f"Génération du profil comparatif pour {track} ({mslot})")

        for v in METEO:
            fig, ax = plt.subplots(figsize=(9, 4))
            legends = []

            for i, (fname, gfile) in enumerate(sub_slot.groupby("fichier_originaire")):
                m = re.search(r"_(\d{8})_(\d{4})", fname)
                if not m:
                    continue
                date_str, time_str = m.groups()
                date_obj = pd.to_datetime(date_str, format="%Y%m%d")
                date_label = date_obj.strftime("%d/%m")

                data = gfile.sort_values("point_id")[["point_id", v]]
                if data[v].isna().all():
                    continue

                x = data["point_id"].values
                y = data[v].values

                color = cmap(i % 10)
                ax.plot(x, y, lw=1.2, alpha=0.9, color=color)
                legends.append(date_label)

            if not legends:
                plt.close(fig)
                continue

            ax.set_title(f"{track.capitalize()} — {v} — {mslot}",
                         fontsize=14, fontweight="bold")
            ax.set_xlabel("point_id")
            ax.set_ylabel(v)
            ax.grid(alpha=0.3)
            ax.legend(legends, loc="upper center", fontsize=8, frameon=False,
                      ncol=5, bbox_to_anchor=(0.5, -0.2))

            outfile = OUT_M / f"profil_compare_{v}_{track}_{mslot}.png"
            fig.tight_layout(rect=[0, 0, 1, 0.92])
            fig.savefig(outfile, dpi=120, bbox_inches="tight")
            plt.close(fig)
            gc.collect()

Même idée ici, mais on vient comparer les différents Mslots sur une même journée

In [ ]:
import re
import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
import gc

COMPARE_D = FIG_DIR / "profil" / "comparer_meme_date"
fc.create_folder(COMPARE_D)

cmap = get_cmap("tab10")  

for track in sorted(bd["track_id"].dropna().unique()):
    sub_track = bd[bd["track_id"] == track]
    OUT_TRACK = COMPARE_D / track
    fc.create_folder(OUT_TRACK)

    dates = []
    for f in sub_track["fichier_originaire"].unique():
        m = re.search(r"_(\d{8})_(\d{4})", f)
        if m:
            dates.append(m.group(1))
    dates = sorted(set(dates))

    for date_str in dates:
        date_obj = pd.to_datetime(date_str, format="%Y%m%d")
        jour = date_obj.day
        mois = date_obj.strftime("%b").lower()  # ex: "oct", "nov"

        OUT_DATE = OUT_TRACK / f"{jour}{mois}"
        fc.create_folder(OUT_DATE)

        print(f"Génération du profil comparatif pour {track} — {jour}{mois}")

        mask_date = bd["fichier_originaire"].str.contains(date_str)
        sub_date = bd[mask_date & (bd["track_id"] == track)]

        for v in METEO:
            fig, ax = plt.subplots(figsize=(9, 4))
            legends = []

            for i, (mslot, gslot) in enumerate(sub_date.groupby("M_slot")):
                data = gslot.sort_values("point_id")[["point_id", v]]
                if v not in data.columns or data[v].isna().all():
                    continue

                x = data["point_id"].values
                y = data[v].values

                color = cmap(i % 10)
                ax.plot(x, y, lw=1.2, alpha=0.9, color=color)
                legends.append(mslot)

            if not legends:
                plt.close(fig)
                continue

            ax.set_title(f"{track.capitalize()} — {v} — {jour}{mois}",
                         fontsize=14, fontweight="bold")
            ax.set_xlabel("point_id")
            ax.set_ylabel(v)
            ax.grid(alpha=0.3)
            ax.legend(legends, loc="upper center", fontsize=8, frameon=False,
                      ncol=5, bbox_to_anchor=(0.5, -0.2))

            outfile = OUT_DATE / f"profil_compare_{v}_{track}_{jour}{mois}.png"
            fig.tight_layout(rect=[0, 0, 1, 0.92])
            fig.savefig(outfile, dpi=120, bbox_inches="tight")
            plt.close(fig)
            gc.collect()

# Barmean des variables météo par parcours

Ce bloc compare la `moyenne des variables météorologiques` entre les trois parcours étudiés : `Antigone`, `Boulevards` et `Écusson`.

In [ ]:
BARMEAN = FIG_DIR / "bar_mean"
fc.create_folder(BARMEAN)

comp = (bd[bd["track_id"].isin(["antigone", "boulevards", "ecusson"])]
          .groupby("track_id")[METEO]
          .mean()
          .T)

for v in comp.index:
    plt.figure(figsize=(6, 4))
    plt.bar(comp.columns.astype(str), comp.loc[v].values)
    plt.title(f"Moyenne par parcours - {v}")
    plt.xlabel("Parcours")
    plt.ylabel(v)
    plt.tight_layout()
    plt.savefig(BARMEAN / f"bar_mean_{v}_parcours.png", dpi=150)